# 00 · Reserve GPU capacity (run before NB1)

## Objective
VTC's scarce GPUs (H100 / B200) aren't on-demand — they come from a **GCE reservation**
you create from your quota. A GPU profile in `configs/cluster_config.yaml` opts in with a
`reservation: <name>` field; NB1 then renders `provisioning_model: RESERVATION` +
`reservation_affinity` pointing at it. This notebook reads that same contract and creates
the reservation(s) it references — so run it **before** `01-provision-cluster.ipynb`.

> ⚠️ **A reservation bills continuously** — reserved H100/B200 cost money whether a cluster
> is using them or not. Create it, run your test, then **delete it** (last cell).

Docs: [Create cluster](https://docs.cloud.google.com/gemini-enterprise-agent-platform/machine-learning/training/training-clusters/create-cluster) ·
[Compute resources](https://docs.cloud.google.com/gemini-enterprise-agent-platform/machine-learning/training/training-clusters/compute-resources)

## 1. Read the contract — which reservations does the config want?

In [1]:
import yaml, subprocess
cfg = yaml.safe_load(open("../configs/cluster_config.yaml"))
PROJECT = cfg["project"]["id"]; ZONE = cfg["project"]["zone"]

# Every fallback_chain profile that sets `reservation:` needs a matching GCE reservation.
RES = [p for p in cfg["fallback_chain"] if p.get("reservation")]
print("project:", PROJECT, "| zone:", ZONE)
for p in RES:
    print(f"  reservation '{p['reservation']}': {p.get('node_count',1)} x {p['machine_type']}  ({p.get('accelerator_type')})")
if not RES:
    print("  (no profile references a reservation — nothing to do; on-demand profiles skip this NB)")

project: sandbox-401718 | zone: us-east4-a
  reservation 'vtc-h100-res': 1 x a3-megagpu-8g  (NVIDIA_H100_MEGA_80GB)


## 2. Create the reservation(s)

`--require-specific-reservation` makes it a **SPECIFIC** reservation (matches the cluster's
`reservation_affinity`). A3/A4 bundle the GPUs into the machine type, so `--accelerator`
isn't usually needed; the cell retries with it if gcloud asks.

**This is also the real quota test:** if create fails with a quota error, your grant
doesn't cover this GPU family — request the matching quota (e.g. `NVIDIA H100 80GB MEGA`).

In [2]:
for p in RES:
    name, mt, n = p["reservation"], p["machine_type"], p.get("node_count", 1)
    base = ["gcloud", "compute", "reservations", "create", name,
            "--project", PROJECT, "--zone", ZONE,
            "--require-specific-reservation", "--vm-count", str(n),
            "--machine-type", mt]
    print(f"== creating reservation '{name}' ({n} x {mt}) ==")
    r = subprocess.run(base)
    if r.returncode and p.get("accelerator_type"):
        acc = p["accelerator_type"].lower().replace("_", "-")   # NVIDIA_H100_MEGA_80GB -> nvidia-h100-mega-80gb
        cnt = p.get("accelerator_count", 8)
        print(f"  retrying with explicit --accelerator=count={cnt},type={acc} ...")
        r = subprocess.run(base + [f"--accelerator=count={cnt},type={acc}"])
    if r.returncode:
        print(f"  (failed for '{name}' — already exists, or a quota error: read the message above)")

== creating reservation 'vtc-h100-res' (1 x a3-megagpu-8g) ==


ERROR: (gcloud.compute.reservations.create) Could not fetch resource:
 - Quota 'GPUS_PER_GPU_FAMILY' exceeded.  Limit: 0.0 in region us-east4.
	metric name = compute.googleapis.com/gpus_per_gpu_family
	limit name = GPUS-PER-GPU-FAMILY-per-project-region
	limit = 0.0
	dimensions = gpu_family: NVIDIA_H100_MEGA
region: us-east4
Try your request in another zone, or view documentation on how to increase quotas: https://cloud.google.com/compute/quotas.



  retrying with explicit --accelerator=count=8,type=nvidia-h100-mega-80gb ...


ERROR: (gcloud.compute.reservations.create) Could not fetch resource:
 - Quota 'GPUS_PER_GPU_FAMILY' exceeded.  Limit: 0.0 in region us-east4.
	metric name = compute.googleapis.com/gpus_per_gpu_family
	limit name = GPUS-PER-GPU-FAMILY-per-project-region
	limit = 0.0
	dimensions = gpu_family: NVIDIA_H100_MEGA
region: us-east4
Try your request in another zone, or view documentation on how to increase quotas: https://cloud.google.com/compute/quotas.



  (failed for 'vtc-h100-res' — already exists, or a quota error: read the message above)


## 3. Verify (should be `READY`)

In [ ]:
import subprocess
subprocess.run(["gcloud", "compute", "reservations", "list", "--project", PROJECT,
                "--zones", ZONE,
                "--format=table(name,status,specificReservation.count,specificReservation.inUseCount)"])

## Next → NB1, then teardown
- Run **`01-provision-cluster.ipynb`** (`MODE=gpu-h100` or `full-chain`) — the `a3h100`
  profile draws from this reservation.
- **When fully done, delete the reservation** (it bills continuously). Run the cell below
  *after* the cluster is torn down (NB3) — a reservation can't be deleted while in use.

In [ ]:
# DANGER: releases the reserved GPU capacity. Run only after the cluster is deleted.
for p in RES:
    name = p["reservation"]
    print(f"== deleting reservation '{name}' ==")
    subprocess.run(["gcloud", "compute", "reservations", "delete", name,
                    "--project", PROJECT, "--zone", ZONE, "--quiet"])